# RBY1 Real Robot Inference with GR00T (Minimal)

OpenPI 실로봇 inference notebook 기준으로, **GR00T 로컬 체크포인트**를 사용하도록 변환한 버전입니다.

## 핵심 차이점 (vs OpenPI)

| 항목 | OpenPI | GR00T |
|------|--------|-------|
| Policy | WebSocket 서버 (`ws_client_policy.infer()`) | 로컬 체크포인트 (`Gr00tPolicy.get_action()`) |
| Action 형식 | **Absolute** joint position (16D) | `decode_action` 내부에서 **Absolute 변환 완료** |
| Internal 표현 | - | Arm: relative delta (model 출력) → processor가 `state[-1] + delta` 로 absolute 변환 |
| Action Horizon | 서버 metadata에서 결정 | 16 steps (rby1_config delta_indices) |
| 변환 필요 | 없음 | 없음 (`policy.get_action()` 반환값이 이미 absolute) |

## 왜 변환이 필요 없는가?

`Gr00tPolicy._get_action()` 내부에서 `decode_action(normalized_action, embodiment_tag, batched_states)`를 호출합니다.  
processor_config에 `use_relative_action=True`가 설정되어 있으므로, `decode_action`이 내부적으로  
`absolute_joint = state[-1] + relative_delta` 변환을 수행합니다.  
따라서 `policy.get_action()` 반환값은 **이미 absolute joint target**이며, 로봇에 직접 전송 가능합니다.

## 연결 구성

| 대상 | 주소 | 비고 |
|------|------|------|
| 실로봇 | `localhost:50051` | 직접 연결 |
| 시뮬레이터 | `localhost:50052` | Docker 포트 매핑 |

### 시뮬레이터 실행 (별도 터미널)
```bash
sudo docker run --rm -it \
  -e DISPLAY=$DISPLAY \
  -v /tmp/.X11-unix:/tmp/.X11-unix \
  -v /exe \
  -p 50052:50051 \
  rainbowroboticsofficial/rby1-sim
```

## 실행 순서
1. 설정/연결 정보 로드 + GR00T Policy 로드
2. 초기 자세(initial pose) 이동 — 실로봇 (safe path + head)
3. Gripper 초기화 (homing + 열기)
4. RealSense 카메라 확인 + env 생성
5. [선택] 시뮬레이터 initial pose + action chunk 테스트
6. Dry-run inference (로봇 명령 없이 출력만 확인)
7. 실로봇 Episode Loop (inference → action chunk 전송 반복)
8. 시각화 + 정리

## 주의
- 시뮬레이터 Docker를 미리 실행한 뒤 진행하세요.
- 실로봇 실행 전 주변 안전을 반드시 확보하세요.

## Step 1 — 라이브러리 Import 및 기본 설정

필요한 라이브러리를 불러오고, GR00T 체크포인트 경로, 로봇 IP, 카메라 시리얼, 제어 파라미터 등 전역 설정값을 정의합니다.

> **항상 가장 먼저 실행해야 하는 셀입니다.**

In [ ]:
from __future__ import annotations

import logging
import sys
import time
from pathlib import Path

import numpy as np
import rby1_sdk as rby

# -- locate Isaac-GR00T root robustly --
_root_candidates = [
    Path("/home/hyunjin/rby1_ws/Isaac-GR00T"),
    Path.cwd(),
    Path.cwd().parent,
]
GROOT_ROOT = None
for _cand in _root_candidates:
    if (_cand / "scripts" / "deployment").exists():
        GROOT_ROOT = _cand.resolve()
        break
if GROOT_ROOT is None:
    raise RuntimeError("Isaac-GR00T root not found. Check workspace path.")

DEPLOYMENT_ROOT = GROOT_ROOT / "scripts" / "deployment"
if str(GROOT_ROOT) not in sys.path:
    sys.path.insert(0, str(GROOT_ROOT))
if str(DEPLOYMENT_ROOT) not in sys.path:
    sys.path.insert(0, str(DEPLOYMENT_ROOT))

from rby1_remote_clients import Gr00tRemotePolicyClient
import rby1_env as _env

logging.basicConfig(level=logging.INFO, force=True)

# ---------- User Config ----------
# real robot / simulator connection (command send is notebook only)
ROBOT_IP = "localhost:50051"
SIM_IP = "localhost:50052"

# external policy service
POLICY_SERVER_HOST = "localhost"
POLICY_SERVER_PORT = 5555

PROMPT = "pick up the cup and place it on the plate"

# RealSense serials
CAM_HEAD_SERIAL = "838212070714"
CAM_LEFT_SERIAL = "922612070040"
CAM_RIGHT_SERIAL = "838212074317"

# test mode -- if True, simulator/single-chunk cells run
TEST_MODE = False

# Runtime/Control
MAX_HZ = 15.0
EPISODE_LENGTH = 400
EXECUTE_CHUNK_SIZE = None  # None이면 predict chunk 전체 실행

ARM_COMMAND_PRIORITY = 10
ARM_MINIMUM_TIME = 10.0
LOG_ACTION_SEND = True
USE_REMOTE_GRIPPER = True

# Initial pose (safe path + head init)
ENABLE_INITIAL_POSE = True
SAFE_INIT_PATH = True
INIT_COMMAND_PRIORITY = 10
INIT_DT = 0.05
INIT_HOLD_TIME = 0.2
ENABLE_HEAD_INIT = True
HEAD_INIT_DEG = np.array([0.0, 40.0], dtype=np.float64)
HEAD_INIT = np.deg2rad(HEAD_INIT_DEG)

print("Config loaded")
print(f"GROOT_ROOT={GROOT_ROOT}")
print(f"ROBOT_IP={ROBOT_IP}  SIM_IP={SIM_IP}")
print(f"POLICY_SERVER={POLICY_SERVER_HOST}:{POLICY_SERVER_PORT}")
print(
    f"CAMERAS head={CAM_HEAD_SERIAL} left={CAM_LEFT_SERIAL} right={CAM_RIGHT_SERIAL}"
)
print(f"PROMPT={PROMPT}")

## Step 1-2 — GR00T Policy 로드 + Helper 함수 정의

GR00T 체크포인트를 로드하고, observation 변환 / action 변환 helper 함수를 정의합니다.

### Action 형식 주의사항
- `right_arm`, `left_arm`: **RELATIVE** delta (현재 관절값에 더해야 함)
- `right_gripper`, `left_gripper`: **ABSOLUTE** 값 (그대로 사용)
- Action horizon: 16 steps

In [ ]:
# ---------- Step 1-2: 원격 GR00T Policy 연결 ----------
policy = Gr00tRemotePolicyClient(
    host=POLICY_SERVER_HOST,
    port=POLICY_SERVER_PORT,
    strict=False,
)

assert policy.ping(), (
    f"policy server is not reachable: {POLICY_SERVER_HOST}:{POLICY_SERVER_PORT}"
)

server_meta = policy.get_server_metadata()
action_keys = list(
    server_meta.get(
        "action_keys",
        ["right_arm", "left_arm", "right_gripper", "left_gripper"],
    )
)
action_horizon = int(server_meta.get("action_horizon", 16))


class _SimpleModalityConfig:
    def __init__(self, keys):
        self.modality_keys = keys


# parse/build helper 호환을 위해 최소 modality 구조 유지
modality = {
    "video": _SimpleModalityConfig(["ego_view", "left_wrist", "right_wrist"]),
    "state": _SimpleModalityConfig(["right_arm", "left_arm", "right_gripper", "left_gripper"]),
    "language": _SimpleModalityConfig(["annotation.human.action.task_description"]),
}

print("Policy server connected.")
print(f"Server metadata: {server_meta}")
print(f"Action keys: {action_keys}")
print(f"Action horizon: {action_horizon}")


# ---------- Helper: (C,H,W) uint8 -> (H,W,C) uint8 ----------
def _chw_to_hwc_uint8(image_chw: np.ndarray) -> np.ndarray:
    image_hwc = np.transpose(image_chw, (1, 2, 0))
    if image_hwc.dtype != np.uint8:
        image_hwc = image_hwc.astype(np.uint8)
    return image_hwc


# ---------- Helper: flat key dict -> GR00T nested input format ----------
# video.*  -> obs["video"][key]   shape (B, T, H, W, C) uint8
# state.*  -> obs["state"][key]   shape (B, T, D) float32
# language -> obs["language"][key] list[list[str]]
def parse_observation_gr00t(obs: dict, modality_cfgs: dict) -> dict:
    new_obs = {}
    for mod in ("video", "state", "language"):
        new_obs[mod] = {}
        for key in modality_cfgs[mod].modality_keys:
            parsed_key = key if mod == "language" else f"{mod}.{key}"
            arr = obs[parsed_key]
            if isinstance(arr, str):
                new_obs[mod][key] = [[arr]]
            else:
                new_obs[mod][key] = arr[None, :]  # add batch dim B=1
    return new_obs


# ---------- Helper: notebook env obs -> GR00T policy input ----------
def build_gr00t_input(env_obs: dict, prompt: str, modality_cfgs: dict) -> dict:
    state16 = np.asarray(env_obs["observation/state"], dtype=np.float32).reshape(-1)
    assert state16.shape[0] == 16, f"Expected 16-dim state, got {state16.shape}"

    head = _chw_to_hwc_uint8(np.asarray(env_obs["observation/head_image"]))
    left = _chw_to_hwc_uint8(np.asarray(env_obs["observation/left_wrist_image"]))
    right = _chw_to_hwc_uint8(np.asarray(env_obs["observation/right_wrist_image"]))

    flat_obs = {
        "state.right_arm": state16[0:7][None, :],
        "state.left_arm": state16[7:14][None, :],
        "state.right_gripper": state16[14:15][None, :],
        "state.left_gripper": state16[15:16][None, :],
        "video.ego_view": head[None, :],
        "video.left_wrist": left[None, :],
        "video.right_wrist": right[None, :],
        "annotation.human.action.task_description": prompt,
    }
    return parse_observation_gr00t(flat_obs, modality_cfgs)


# ---------- Helper: GR00T action dict -> (horizon, 16) absolute-target array ----------
# policy.get_action() 반환값은 decode_action(use_relative_action=True) 내부에서
# 이미 absolute joint target으로 변환되어 있습니다. 로봇에 직접 전송 가능.
# Column order: [right_arm(7), left_arm(7), right_gripper(1), left_gripper(1)]
def flatten_action_chunk(action_chunk: dict, act_keys: list) -> np.ndarray:
    unbatched = {k: np.asarray(v)[0] for k, v in action_chunk.items()}  # remove batch dim
    horizon = len(unbatched[act_keys[0]])
    traj = []
    for t in range(horizon):
        step = np.concatenate(
            [np.atleast_1d(unbatched[k][t]) for k in act_keys], axis=0
        )
        traj.append(step.astype(np.float32))
    return np.array(traj, dtype=np.float32)


print("Helper functions defined.")
print("  build_gr00t_input()    : observation -> GR00T input")
print("  flatten_action_chunk() : GR00T action dict -> (T, 16) ABSOLUTE target array")

## Step 2 — 실로봇 초기 자세 이동

로봇 팔을 안전한 waypoint 경로를 거쳐 데이터 수집 시작 자세로 이동시킵니다.

- **elbow 굽힘 여부**에 따라 짧은 경로(midpoint2 → initial) 또는 전체 safe path(midpoint1 → midpoint2 → initial)를 선택합니다.
- Head 초기화가 활성화되면 지정 각도로 이동합니다.
- Tool flange 12V를 공급하여 그리퍼 전원을 켭니다.

In [ ]:
# ---------- Step 2: initial pose 이동 (safe path + head) ----------
if ENABLE_INITIAL_POSE:
    # data-collection 초기 자세
    TORSO_INIT_DEG = np.array([0.0, 20.0, -40.0, 35.0, 0.0, 0.0], dtype=np.float64)
    INIT_RIGHT_DEG = np.array([-24.0, -60.0, 10.0, -120.0, -60.0, 85.0, 0.0], dtype=np.float64)
    INIT_LEFT_DEG = np.array([-24.0, 60.0, -10.0, -120.0, 60.0, 85.0, 0.0], dtype=np.float64)
    TORSO_INIT = np.deg2rad(TORSO_INIT_DEG)
    INIT_RIGHT = np.deg2rad(INIT_RIGHT_DEG)
    INIT_LEFT = np.deg2rad(INIT_LEFT_DEG)

    MID1_RIGHT_DEG = np.array([70.0, -30.0, 0.0, -100.0, 0.0, -70.0, 0.0], dtype=np.float64)
    MID1_LEFT_DEG = np.array([70.0, 30.0, 0.0, -100.0, 0.0, -70.0, 0.0], dtype=np.float64)
    MID2_RIGHT_DEG = np.array([0.0, -15.0, 0.0, -100.0, 0.0, -70.0, 0.0], dtype=np.float64)
    MID2_LEFT_DEG = np.array([0.0, 15.0, 0.0, -100.0, 0.0, -70.0, 0.0], dtype=np.float64)
    MID1_RIGHT = np.deg2rad(MID1_RIGHT_DEG)
    MID1_LEFT = np.deg2rad(MID1_LEFT_DEG)
    MID2_RIGHT = np.deg2rad(MID2_RIGHT_DEG)
    MID2_LEFT = np.deg2rad(MID2_LEFT_DEG)

    TORSO_SLICE = slice(2, 8)
    RIGHT_SLICE = slice(8, 15)
    LEFT_SLICE = slice(15, 22)
    ELBOW_INIT_THRESHOLD_DEG = 90.0

    def _build_body_head_cmd(torso_q, right_q, left_q, head_q, dt):
        body_builder = rby.BodyComponentBasedCommandBuilder()
        if torso_q is not None:
            body_builder = body_builder.set_torso_command(
                rby.JointPositionCommandBuilder()
                .set_command_header(rby.CommandHeaderBuilder().set_control_hold_time(INIT_HOLD_TIME))
                .set_position(np.asarray(torso_q, dtype=np.float64))
                .set_minimum_time(float(dt))
            )
        body_builder = (
            body_builder
            .set_right_arm_command(
                rby.JointPositionCommandBuilder()
                .set_command_header(rby.CommandHeaderBuilder().set_control_hold_time(INIT_HOLD_TIME))
                .set_position(np.asarray(right_q, dtype=np.float64))
                .set_minimum_time(float(dt))
            )
            .set_left_arm_command(
                rby.JointPositionCommandBuilder()
                .set_command_header(rby.CommandHeaderBuilder().set_control_hold_time(INIT_HOLD_TIME))
                .set_position(np.asarray(left_q, dtype=np.float64))
                .set_minimum_time(float(dt))
            )
        )
        cbc = rby.ComponentBasedCommandBuilder().set_body_command(body_builder)
        if head_q is not None:
            cbc = cbc.set_head_command(
                rby.JointPositionCommandBuilder()
                .set_command_header(rby.CommandHeaderBuilder().set_control_hold_time(INIT_HOLD_TIME))
                .set_position(np.asarray(head_q, dtype=np.float64))
                .set_minimum_time(float(dt))
            )
        return rby.RobotCommandBuilder().set_command(cbc)

    def _move_waypoint_stream(robot, stream, name, torso_target, right_target, left_target, head_target, duration):
        q0 = np.asarray(robot.get_state().position, dtype=np.float64)
        start_torso = q0[TORSO_SLICE].copy()
        start_right = q0[RIGHT_SLICE].copy()
        start_left = q0[LEFT_SLICE].copy()
        goal_torso = torso_target if torso_target is not None else start_torso
        steps = max(2, int(np.ceil(duration / INIT_DT)))
        torso_path = np.linspace(start_torso, goal_torso, num=steps)
        right_path = np.linspace(start_right, right_target, num=steps)
        left_path = np.linspace(start_left, left_target, num=steps)
        for i in range(steps):
            cmd = _build_body_head_cmd(torso_path[i], right_path[i], left_path[i], head_target, dt=INIT_DT)
            try:
                stream.send_command(cmd)
            except RuntimeError as exc:
                if "expired" in str(exc).lower():
                    robot.wait_for_control_ready(1000)
                    stream = robot.create_command_stream(priority=INIT_COMMAND_PRIORITY)
                    stream.send_command(cmd)
                else:
                    raise
            time.sleep(INIT_DT)
        q_now = np.asarray(robot.get_state().position, dtype=np.float64)
        print(f"[init-pose] reached {name} | torso_now[1]={q_now[3]:+.4f}, right_now[0]={q_now[8]:+.4f}, left_now[0]={q_now[15]:+.4f}")
        return stream

    robot_init = rby.create_robot(ROBOT_IP, "a") if hasattr(rby, "create_robot") else rby.create_robot_a(ROBOT_IP)
    robot_init.connect()
    assert robot_init.is_connected(), f"Failed to connect robot at {ROBOT_IP}"

    robot_init.power_on(".*")
    robot_init.servo_on(".*")
    robot_init.enable_control_manager()

    try:
        robot_init.cancel_control()
    except Exception:
        pass
    robot_init.wait_for_control_ready(1000)

    q0 = np.asarray(robot_init.get_state().position, dtype=np.float64)
    print("[init-pose] current torso:", np.round(q0[TORSO_SLICE], 3))
    print("[init-pose] current right:", np.round(q0[RIGHT_SLICE], 3))
    print("[init-pose] current left :", np.round(q0[LEFT_SLICE], 3))
    if ENABLE_HEAD_INIT:
        print("[init-pose] target head(deg):", np.round(HEAD_INIT_DEG, 2))

    right_elbow_now = float(q0[RIGHT_SLICE][3])
    left_elbow_now  = float(q0[LEFT_SLICE][3])
    elbow_bent = (
        abs(right_elbow_now) > np.deg2rad(ELBOW_INIT_THRESHOLD_DEG)
        or abs(left_elbow_now) > np.deg2rad(ELBOW_INIT_THRESHOLD_DEG)
    )
    print(
        f"[init-pose] elbow check | right={np.degrees(right_elbow_now):+.1f}, "
        f"left={np.degrees(left_elbow_now):+.1f}  "
        f"-> {'near-INITIAL (bent)' if elbow_bent else 'near-ZERO (straight)'}"
    )

    if SAFE_INIT_PATH:
        if elbow_bent:
            waypoints = [
                ("midpoint2", None,       MID2_RIGHT, MID2_LEFT, 5.0),
                ("initial",   TORSO_INIT, INIT_RIGHT, INIT_LEFT, 3.0),
            ]
            print("[init-pose] near-initial detected: midpoint2 -> initial")
        else:
            waypoints = [
                ("midpoint1", None,       MID1_RIGHT, MID1_LEFT, 5.0),
                ("midpoint2", None,       MID2_RIGHT, MID2_LEFT, 5.0),
                ("initial",   TORSO_INIT, INIT_RIGHT, INIT_LEFT, 3.0),
            ]
            print("[init-pose] near-zero detected: midpoint1 -> midpoint2 -> initial")
    else:
        waypoints = [("initial", TORSO_INIT, INIT_RIGHT, INIT_LEFT, 4.0)]
        print("[init-pose] SAFE_INIT_PATH disabled: direct -> initial")

    head_target = HEAD_INIT if ENABLE_HEAD_INIT else None
    stream = robot_init.create_command_stream(priority=INIT_COMMAND_PRIORITY)
    for name, torso_target, right_target, left_target, duration in waypoints:
        print(f"[init-pose] move -> {name} (t={duration:.1f}s)")
        stream = _move_waypoint_stream(
            robot_init, stream, name=name,
            torso_target=torso_target, right_target=right_target,
            left_target=left_target, head_target=head_target, duration=duration,
        )

    time.sleep(0.2)
    q_end = np.asarray(robot_init.get_state().position, dtype=np.float64)
    err_t = float(np.linalg.norm(q_end[TORSO_SLICE] - TORSO_INIT))
    err_r = float(np.linalg.norm(q_end[RIGHT_SLICE] - INIT_RIGHT))
    err_l = float(np.linalg.norm(q_end[LEFT_SLICE] - INIT_LEFT))
    print(f"[init-pose] done | final error norm torso={err_t:.6f}, right={err_r:.6f}, left={err_l:.6f}")

    if USE_REMOTE_GRIPPER:
        for _arm in ("right", "left"):
            ok_v = robot_init.set_tool_flange_output_voltage(_arm, 12)
            print(f"[init-pose] tool flange 12V {_arm}: {'OK' if ok_v else 'FAILED'}")
        time.sleep(0.5)

    try:
        robot_init.cancel_control()
    except Exception:
        pass
    if hasattr(robot_init, "disconnect"):
        robot_init.disconnect()

    INITIAL_POSE_DONE = True
else:
    INITIAL_POSE_DONE = False
    print("[init-pose] skipped")

## Step 2-5 — Gripper 초기화 (Homing + 열기)

RemoteGripper UDP 클라이언트를 통해 그리퍼를 초기화합니다.

1. `initialize()` — 서버 ping 확인
2. `homing()` — 범위 캘리브레이션 (min_q / max_q)
3. `start()` — 제어 루프 시작
4. 완전 열기 (normalized=1.0)

In [ ]:
# ---------- Step 2-5: Gripper 초기화 (homing + 열기) ----------
# 전제: Step 2 (initial pose) 실행 완료 -> tool flange 12V 공급 완료
if not globals().get("INITIAL_POSE_DONE", False):
    raise RuntimeError("먼저 Step 2 (initial pose) 셀을 실행하세요.")
if not globals().get("USE_REMOTE_GRIPPER", False):
    print("[gripper-init] USE_REMOTE_GRIPPER=False -> skip")
else:
    from rby1_remote_gripper import Gripper as RemoteGripper

    # Step 2에서 남아있는 stream 잔여 객체 제거
    # stream이 살아있으면 priority=10 control을 잡고 있어 이후 env 명령을 차단할 수 있음
    if "stream" in globals() and stream is not None:
        try:
            del stream
        except Exception:
            pass
        stream = None
        print("[gripper-init] 이전 stream 객체 정리 완료")

    # 이전 gripper 객체가 있으면 정리
    if "gripper_obj" in globals() and gripper_obj is not None:
        try:
            gripper_obj.stop()
            print("[gripper-init] 이전 gripper 객체 stop() 완료")
        except Exception as _e:
            print(f"[gripper-init] 이전 gripper stop 무시: {_e}")
        gripper_obj = None

    print("[gripper-init] RemoteGripper 연결 중...")
    gripper_obj = RemoteGripper()
    print(f"[gripper-init] host={gripper_obj.host}  port={gripper_obj.port}")

    if gripper_obj.host is None or gripper_obj.port is None:
        raise RuntimeError(
            "[gripper-init] gripper host/port가 설정되지 않았습니다.\n"
            "  scripts/deployment/config.yaml 또는 환경변수 확인"
        )

    # 1) ping / initialize
    print("[gripper-init] ping 확인 중...")
    ok_init = gripper_obj.initialize(verbose=True)
    print(f"[gripper-init] ping 결과: {'OK' if ok_init else 'FAILED'}")
    if not ok_init:
        raise RuntimeError("[gripper-init] 그리퍼 서버 응답 없음.")

    # 2) homing (범위 캘리브레이션)
    print("[gripper-init] homing 중... (30초 이내)")
    ok_homing = gripper_obj.homing()
    if not ok_homing:
        raise RuntimeError("[gripper-init] homing 실패")

    # homing 결과로 얻은 min_q / max_q 확인
    _min_q = getattr(gripper_obj, "min_q", None)
    _max_q = getattr(gripper_obj, "max_q", None)
    if _min_q is None or _max_q is None:
        raise RuntimeError(
            "[gripper-init] homing 성공했지만 min_q/max_q가 없음.\n"
            "  gripper_server.py 응답 형식에 min_q/max_q 필드가 있는지 확인하세요."
        )
    print(f"[gripper-init] homing 완료 | min_q={_min_q}  max_q={_max_q}")

    # 3) 제어 루프 시작
    print("[gripper-init] start() 호출 중...")
    gripper_obj.start()
    print("[gripper-init] start() 완료")

    # 4) 완전 열기/상태확인 구간 timeout 단축
    _old_timeout = getattr(gripper_obj, "timeout", None)
    _fast_timeout = 3.0
    try:
        _old_timeout_f = float(_old_timeout) if _old_timeout is not None else None
    except Exception:
        _old_timeout_f = None
    if _old_timeout_f is None or _old_timeout_f > _fast_timeout:
        gripper_obj.timeout = _fast_timeout
        try:
            gripper_obj._sock.settimeout(_fast_timeout)
        except Exception:
            pass
    print(f"[gripper-init] fast timeout 적용: {_old_timeout} -> {gripper_obj.timeout}")

    # 완전 열기 (normalized_q=1.0 -> OPEN) -- wait_for_reply=True로 확인
    print("[gripper-init] 완전 열기 명령 전송 (normalized=1.0)...")
    gripper_obj.set_normalized_target(np.array([1.0, 1.0]), wait_for_reply=True)
    time.sleep(0.5)

    # 현재 그리퍼 상태 확인
    try:
        _grip_state = gripper_obj.get_state()
        print(f"[gripper-init] 현재 그리퍼 상태: {_grip_state}  (열기 후 확인)")
    except Exception as _se:
        print(f"[gripper-init][WARN] 상태 조회 실패: {_se}")

    # 현재 normalized target 확인
    try:
        _grip_norm = gripper_obj.get_normalized_target()
        print(f"[gripper-init] 현재 normalized target: {_grip_norm}  (1.0=OPEN 이어야 함)")
    except Exception as _ne:
        print(f"[gripper-init][WARN] normalized target 조회 실패: {_ne}")

    GRIPPER_INIT_DONE = True
    print("[gripper-init] 완료")

## Step 3 — RealSense 카메라 초기화 + 환경(env) 구성

`MultiRealsense`를 이용해 Head / Left Wrist / Right Wrist 3대의 카메라를 동시에 사용합니다.
OpenPI의 `RBY1Environment`를 카메라 + observation 래퍼로 재활용합니다 (policy 서버 연결 없음).

> GR00T는 서버가 아닌 로컬 inference이므로 `runtime`은 생성하지 않습니다.

In [ ]:
# ---------- Step 3: notebook 내부 observation 구성 ----------
import gc
import importlib
import matplotlib.pyplot as plt
import pyrealsense2 as rs

# (A) RealSense 장치 진단
CAM_SERIALS = {
    "head": CAM_HEAD_SERIAL,
    "left_wrist": CAM_LEFT_SERIAL,
    "right_wrist": CAM_RIGHT_SERIAL,
}

IMG_WIDTH = 640
IMG_HEIGHT = 480

_ctx = rs.context()
_available = {
    dev.get_info(rs.camera_info.serial_number): dev.get_info(rs.camera_info.name)
    for dev in _ctx.query_devices()
}
del _ctx

print("-" * 60)
print(f"  [camera-check] connected devices: {len(_available)}")
for serial, name in _available.items():
    cfg_name = next((n for n, s in CAM_SERIALS.items() if s == serial), "unknown")
    mark = "OK" if serial in CAM_SERIALS.values() else "INFO"
    print(f"  {mark:4s} {cfg_name:>12} | {name} | S/N: {serial}")

_not_found = [(n, s) for n, s in CAM_SERIALS.items() if s not in _available]
if _not_found:
    for n, s in _not_found:
        print(f"  MISSING {n}: {s}")
    raise RuntimeError("Some configured cameras are not detected. Check USB connections.")
print(f"  OK: all configured cameras detected ({len(CAM_SERIALS)})")
print("-" * 60)

if not _available:
    raise RuntimeError("No RealSense camera detected.")

# (B) 기존 env 정리
if "env" in globals() and env is not None:
    try:
        env.close()
        print("[cleanup] 기존 env 닫힘")
    except Exception as _e:
        print(f"[cleanup] 기존 env 닫기 실패 (무시): {_e}")
    env = None

gc.collect()
time.sleep(1.5)

# (C) notebook 내부에서 env 생성
_env = importlib.reload(_env)
env = _env.RBY1Environment(
    robot_ip=ROBOT_IP,
    prompt=PROMPT,
    render_height=224,
    render_width=224,
    camera_width=IMG_WIDTH,
    camera_height=IMG_HEIGHT,
    camera_fps=30,
    cam_head_serial=CAM_HEAD_SERIAL,
    cam_left_serial=CAM_LEFT_SERIAL,
    cam_right_serial=CAM_RIGHT_SERIAL,
    left_action_dim=8,
    right_action_dim=8,
    arm_command_priority=ARM_COMMAND_PRIORITY,
    arm_minimum_time=ARM_MINIMUM_TIME,
    log_action_send=LOG_ACTION_SEND,
    state_source="robot",
    state_zmq_address=None,
    state_indices=None,
    gripper_state_key=None,
    use_remote_gripper=USE_REMOTE_GRIPPER,
    gripper=globals().get("gripper_obj"),
)

obs = env.get_observation()
print(
    "[obs] image shapes:",
    {k: tuple(v.shape) for k, v in obs.items() if isinstance(v, np.ndarray) and "image" in k},
)
state_shape = np.asarray(obs["observation/state"]).shape
print("[obs] state shape:", state_shape)
if state_shape != (16,):
    raise RuntimeError(
        f"Expected observation/state shape (16,), got {state_shape}. "
        "Check gripper init/state source."
    )

# (D) 첫 관측 이미지 시각화
_img_keys = [k for k in sorted(obs.keys()) if "image" in k and isinstance(obs[k], np.ndarray)]
if _img_keys:
    _fig, _axes = plt.subplots(1, len(_img_keys), figsize=(5 * len(_img_keys), 5))
    if len(_img_keys) == 1:
        _axes = [_axes]
    for _ax, _k in zip(_axes, _img_keys):
        _img = obs[_k]
        if _img.ndim == 3 and _img.shape[0] in (1, 3):
            _img = np.transpose(_img, (1, 2, 0))
        _ax.imshow(_img)
        _ax.set_title(_k.replace("observation/", ""), fontsize=11)
        _ax.axis("off")
    _fig.suptitle("Initial Observation (from RBY1Environment)", fontsize=13)
    plt.tight_layout()
    plt.show()

ENV_SETUP_DONE = True
print("[setup] 완료: notebook env 연결 + 관측 확인")

---
## [선택] 시뮬레이터 초기 자세 이동

`SIM_IP`에서 실행 중인 시뮬레이터 로봇을 실로봇과 동일한 초기 자세로 이동시킵니다.
실로봇 inference 전에 시뮬레이터로 먼저 동작을 검증할 때 사용합니다.

> `TEST_MODE=False`이면 자동으로 건너뜁니다.

In [ ]:
# ---------- 시뮬레이터 initial pose 이동 ----------
if not globals().get("TEST_MODE", False):
    print("[sim-init] TEST_MODE=False -> skip")
else:
    SIM_ADDR = SIM_IP
    SIM_INIT_DT = 0.05
    SIM_INIT_HOLD_TIME = 0.5

    TORSO_INIT_DEG_SIM = np.array([0.0, 20.0, -40.0, 35.0, 0.0, 0.0], dtype=np.float64)
    INIT_RIGHT_DEG_SIM = np.array([-24.0, -60.0, 10.0, -120.0, -60.0, 85.0, 0.0], dtype=np.float64)
    INIT_LEFT_DEG_SIM  = np.array([-24.0,  60.0, -10.0, -120.0,  60.0, 85.0, 0.0], dtype=np.float64)
    MID1_RIGHT_DEG_SIM = np.array([70.0, -30.0, 0.0, -100.0, 0.0, -70.0, 0.0], dtype=np.float64)
    MID1_LEFT_DEG_SIM  = np.array([70.0,  30.0, 0.0, -100.0, 0.0, -70.0, 0.0], dtype=np.float64)
    MID2_RIGHT_DEG_SIM = np.array([0.0,  -15.0, 0.0, -100.0, 0.0, -70.0, 0.0], dtype=np.float64)
    MID2_LEFT_DEG_SIM  = np.array([0.0,   15.0, 0.0, -100.0, 0.0, -70.0, 0.0], dtype=np.float64)

    TORSO_INIT_SIM = np.deg2rad(TORSO_INIT_DEG_SIM)
    INIT_RIGHT_SIM = np.deg2rad(INIT_RIGHT_DEG_SIM)
    INIT_LEFT_SIM  = np.deg2rad(INIT_LEFT_DEG_SIM)
    MID1_RIGHT_SIM = np.deg2rad(MID1_RIGHT_DEG_SIM)
    MID1_LEFT_SIM  = np.deg2rad(MID1_LEFT_DEG_SIM)
    MID2_RIGHT_SIM = np.deg2rad(MID2_RIGHT_DEG_SIM)
    MID2_LEFT_SIM  = np.deg2rad(MID2_LEFT_DEG_SIM)

    TORSO_SLICE_SIM = slice(2, 8)
    RIGHT_SLICE_SIM = slice(8, 15)
    LEFT_SLICE_SIM  = slice(15, 22)

    def _build_body_cmd_sim(torso_q, right_q, left_q, dt):
        body_builder = rby.BodyComponentBasedCommandBuilder()
        if torso_q is not None:
            body_builder = body_builder.set_torso_command(
                rby.JointPositionCommandBuilder()
                .set_command_header(rby.CommandHeaderBuilder().set_control_hold_time(SIM_INIT_HOLD_TIME))
                .set_position(np.asarray(torso_q, dtype=np.float64))
                .set_minimum_time(float(dt))
            )
        body_builder = (
            body_builder
            .set_right_arm_command(
                rby.JointPositionCommandBuilder()
                .set_command_header(rby.CommandHeaderBuilder().set_control_hold_time(SIM_INIT_HOLD_TIME))
                .set_position(np.asarray(right_q, dtype=np.float64))
                .set_minimum_time(float(dt))
            )
            .set_left_arm_command(
                rby.JointPositionCommandBuilder()
                .set_command_header(rby.CommandHeaderBuilder().set_control_hold_time(SIM_INIT_HOLD_TIME))
                .set_position(np.asarray(left_q, dtype=np.float64))
                .set_minimum_time(float(dt))
            )
        )
        return rby.RobotCommandBuilder().set_command(
            rby.ComponentBasedCommandBuilder().set_body_command(body_builder)
        )

    def _move_waypoint_sim(robot, stream, torso_target, right_target, left_target, duration):
        q0 = np.asarray(robot.get_state().position, dtype=np.float64)
        start_torso = q0[TORSO_SLICE_SIM].copy()
        start_right = q0[RIGHT_SLICE_SIM].copy()
        start_left  = q0[LEFT_SLICE_SIM].copy()
        goal_torso = torso_target if torso_target is not None else start_torso
        steps = max(2, int(np.ceil(duration / SIM_INIT_DT)))
        torso_path = np.linspace(start_torso, goal_torso, num=steps)
        right_path = np.linspace(start_right, right_target, num=steps)
        left_path  = np.linspace(start_left,  left_target,  num=steps)
        for i in range(steps):
            cmd_wp = _build_body_cmd_sim(torso_path[i], right_path[i], left_path[i], dt=SIM_INIT_DT)
            try:
                stream.send_command(cmd_wp)
            except RuntimeError as exc:
                if "expired" in str(exc).lower():
                    robot.wait_for_control_ready(1000)
                    stream = robot.create_command_stream(priority=10)
                    stream.send_command(cmd_wp)
                else:
                    raise
            time.sleep(SIM_INIT_DT)
        return stream

    robot_sim = rby.create_robot_a(SIM_ADDR) if hasattr(rby, "create_robot_a") else rby.create_robot(SIM_ADDR, "a")
    robot_sim.connect()
    assert robot_sim.is_connected(), f"Failed to connect simulator at {SIM_ADDR}"

    robot_sim.power_on(".*")
    robot_sim.servo_on(".*")
    robot_sim.reset_fault_control_manager()
    if not robot_sim.enable_control_manager():
        raise RuntimeError("[sim-init] Failed to enable control manager")

    stream_sim = robot_sim.create_command_stream(priority=10)
    sim_waypoints = [
        (None,           MID1_RIGHT_SIM, MID1_LEFT_SIM, 5.0),
        (None,           MID2_RIGHT_SIM, MID2_LEFT_SIM, 5.0),
        (TORSO_INIT_SIM, INIT_RIGHT_SIM, INIT_LEFT_SIM, 3.0),
    ]
    for torso_t, right_t, left_t, dur_t in sim_waypoints:
        stream_sim = _move_waypoint_sim(robot_sim, stream_sim, torso_t, right_t, left_t, dur_t)

    time.sleep(0.2)
    q_init = np.asarray(robot_sim.get_state().position, dtype=np.float64)
    print("[sim-init] q_init right:", np.round(q_init[8:15], 4))
    print("[sim-init] q_init left :", np.round(q_init[15:22], 4))

    try:
        robot_sim.cancel_control()
    except Exception:
        pass
    if hasattr(robot_sim, "disconnect"):
        robot_sim.disconnect()

    SIM_INITIAL_POSE_DONE = True
    print("[sim-init] done")

## [선택] 시뮬레이터 단일 Action Chunk 전송 테스트

GR00T inference를 1회 수행하고, action chunk를 시뮬레이터로 전송합니다.
`policy.get_action()` 반환값은 이미 absolute target입니다 (decode_action 내부 변환).
실로봇에 전송하기 전 policy 출력이 안전한지 확인하는 용도입니다.

> `TEST_MODE=False`이면 자동으로 건너뜁니다.

In [ ]:
# ---------- GR00T action chunk -> 시뮬레이터 전송 ----------
if not globals().get("TEST_MODE", False):
    print("[sim-chunk] TEST_MODE=False -> skip")
else:
    if not globals().get("ENV_SETUP_DONE", False):
        raise RuntimeError("먼저 Step 3 셀을 실행하세요.")
    if not globals().get("SIM_INITIAL_POSE_DONE", False):
        raise RuntimeError("먼저 시뮬레이터 initial pose 셀을 실행하세요.")

    SIM_ADDR            = SIM_IP
    SIM_CHUNK_DT        = 0.1
    SIM_CHUNK_HOLD_TIME = 0.2
    SIM_CHUNK_PRIORITY  = 10

    # 1) inference
    # policy.get_action() 반환값은 decode_action(use_relative_action=True) 내부에서
    # 이미 absolute target으로 변환된 값입니다.
    obs_sim = env.get_observation()
    parsed_obs_sim = build_gr00t_input(obs_sim, PROMPT, modality)
    action_chunk_sim, _ = policy.get_action(parsed_obs_sim)
    print(f"actio dim: {action_chunk_sim.keys()} | expected keys: {action_keys}")
    print(f"shape: {[v.shape for v in action_chunk_sim.values()]}")
    traj16_sim = flatten_action_chunk(action_chunk_sim, action_keys)  # already absolute

    right_targets = traj16_sim[:, 0:7]
    left_targets  = traj16_sim[:, 7:14]

    print(f"[sim-chunk] action chunk shape: {traj16_sim.shape}")
    print(f"[sim-chunk] right target[0] : {np.round(right_targets[0], 4)}")
    print(f"[sim-chunk] right target[-1]: {np.round(right_targets[-1], 4)}")
    print(f"[sim-chunk] left  target[0] : {np.round(left_targets[0], 4)}")
    print(f"[sim-chunk] left  target[-1]: {np.round(left_targets[-1], 4)}")

    total_cmd_norm_r = float(np.linalg.norm(right_targets[-1] - right_targets[0]))
    total_cmd_norm_l = float(np.linalg.norm(left_targets[-1]  - left_targets[0]))
    print(f"[sim-chunk] cmd movement norm | right={total_cmd_norm_r:.4f}, left={total_cmd_norm_l:.4f}")
    if total_cmd_norm_r < 1e-3 and total_cmd_norm_l < 1e-3:
        print("[sim-chunk][WARN] 모델 output 변화가 매우 작습니다. 관측이 올바른지 확인하세요.")

    # 2) 시뮬레이터 연결
    robot_sim = rby.create_robot_a(SIM_ADDR) if hasattr(rby, "create_robot_a") else rby.create_robot(SIM_ADDR, "a")
    robot_sim.connect()
    assert robot_sim.is_connected()

    robot_sim.power_on(".*")
    robot_sim.servo_on(".*")
    robot_sim.reset_fault_control_manager()
    if not robot_sim.enable_control_manager():
        raise RuntimeError("[sim-chunk] Failed to enable control manager")

    q0_sim      = np.asarray(robot_sim.get_state().position, dtype=np.float64)
    torso_hold  = q0_sim[2:8].copy()
    start_r_sim = q0_sim[8:15].copy()
    start_l_sim = q0_sim[15:22].copy()

    # 3) stream 전송
    stream_sim  = robot_sim.create_command_stream(priority=SIM_CHUNK_PRIORITY)
    send_errors = 0

    for i in range(traj16_sim.shape[0]):
        cmd = rby.RobotCommandBuilder().set_command(
            rby.ComponentBasedCommandBuilder().set_body_command(
                rby.BodyComponentBasedCommandBuilder()
                .set_torso_command(
                    rby.JointPositionCommandBuilder()
                    .set_command_header(rby.CommandHeaderBuilder().set_control_hold_time(SIM_CHUNK_HOLD_TIME))
                    .set_position(torso_hold.astype(np.float64))
                    .set_minimum_time(SIM_CHUNK_DT)
                )
                .set_right_arm_command(
                    rby.JointPositionCommandBuilder()
                    .set_command_header(rby.CommandHeaderBuilder().set_control_hold_time(SIM_CHUNK_HOLD_TIME))
                    .set_position(right_targets[i].astype(np.float64))
                    .set_minimum_time(SIM_CHUNK_DT)
                )
                .set_left_arm_command(
                    rby.JointPositionCommandBuilder()
                    .set_command_header(rby.CommandHeaderBuilder().set_control_hold_time(SIM_CHUNK_HOLD_TIME))
                    .set_position(left_targets[i].astype(np.float64))
                    .set_minimum_time(SIM_CHUNK_DT)
                )
            )
        )

        try:
            stream_sim.send_command(cmd)
        except RuntimeError as exc:
            send_errors += 1
            if "expired" in str(exc).lower():
                robot_sim.wait_for_control_ready(1000)
                stream_sim = robot_sim.create_command_stream(priority=SIM_CHUNK_PRIORITY)
                stream_sim.send_command(cmd)
            else:
                raise

        if i == 0 or (i + 1) % 5 == 0 or (i + 1) == traj16_sim.shape[0]:
            q_now = np.asarray(robot_sim.get_state().position, dtype=np.float64)
            err_r   = float(np.linalg.norm(right_targets[i] - q_now[8:15]))
            err_l   = float(np.linalg.norm(left_targets[i]  - q_now[15:22]))
            moved_r = float(np.linalg.norm(q_now[8:15]  - start_r_sim))
            moved_l = float(np.linalg.norm(q_now[15:22] - start_l_sim))
            print(
                f"[sim-chunk] step {i+1:3d}/{traj16_sim.shape[0]} "
                f"| tracking_err(r={err_r:.4f}, l={err_l:.4f}) "
                f"| moved(r={moved_r:.4f}, l={moved_l:.4f})"
            )

        time.sleep(SIM_CHUNK_DT)

    time.sleep(0.5)
    q_after = np.asarray(robot_sim.get_state().position, dtype=np.float64)
    obs_norm_r = float(np.linalg.norm(q_after[8:15]  - start_r_sim))
    obs_norm_l = float(np.linalg.norm(q_after[15:22] - start_l_sim))
    print(f"\n[sim-chunk] === 결과 ===")
    print(f"[sim-chunk] observed movement norm | right={obs_norm_r:.4f}, left={obs_norm_l:.4f}")
    print(f"[sim-chunk] send errors: {send_errors}")
    if obs_norm_r < 1e-3 and obs_norm_l < 1e-3:
        print("[sim-chunk][WARN] 실제 이동량이 거의 없음. 관측 형식을 확인하세요.")

    try:
        robot_sim.cancel_control()
    except Exception:
        pass
    if hasattr(robot_sim, "disconnect"):
        robot_sim.disconnect()

## Step 4 — Dry-run Inference (로봇 명령 없음)

GR00T inference를 1회 수행하고 출력만 확인합니다. 로봇에는 아무 명령도 보내지 않습니다.

**출력값 해석:**
- `policy.get_action()` 반환값은 `decode_action(use_relative_action=True)` 덕분에 이미 **absolute joint target**입니다.
- 현재 state와 가깝게 clipping된 궤적이 나오면 정상입니다 (relative delta가 작으면 현재 위치 근처 target이 됨).

In [ ]:
# ---------- Step 4: Dry-run inference ----------
if not globals().get("ENV_SETUP_DONE", False):
    raise RuntimeError("먼저 Step 3 셀을 실행하세요.")

obs_dry = env.get_observation()
parsed_dry = build_gr00t_input(obs_dry, PROMPT, modality)

action_chunk_dry, _ = policy.get_action(parsed_dry)
traj16_dry = flatten_action_chunk(action_chunk_dry, action_keys)  # already absolute

current_state16 = np.asarray(obs_dry["observation/state"], dtype=np.float32).reshape(-1)
print(f"[dry-run] action chunk shape: {traj16_dry.shape}")
print(f"[dry-run] current state (rad): {np.round(current_state16, 4)}")
print()
print(f"[dry-run] absolute right[0] : {np.round(traj16_dry[0, 0:7], 4)}")
print(f"[dry-run] absolute right[-1]: {np.round(traj16_dry[-1, 0:7], 4)}")
print(f"[dry-run] absolute left[0]  : {np.round(traj16_dry[0, 7:14], 4)}")
print(f"[dry-run] absolute left[-1] : {np.round(traj16_dry[-1, 7:14], 4)}")
print()

# 현재 state와의 차이 (de-facto relative delta) 확인용
delta_check = traj16_dry.copy()
delta_check[:, 0:7]  -= current_state16[0:7]
delta_check[:, 7:14] -= current_state16[7:14]
print(f"[dry-run] implied delta right[0] : {np.round(delta_check[0, 0:7], 4)}")
print(f"[dry-run] implied delta norm per step:")
for t in range(traj16_dry.shape[0]):
    dr = float(np.linalg.norm(delta_check[t, 0:7]))
    dl = float(np.linalg.norm(delta_check[t, 7:14]))
    if t == 0 or t == traj16_dry.shape[0] - 1 or (t + 1) % 4 == 0:
        print(f"  step {t:2d}: delta_norm right={dr:.4f}, left={dl:.4f} "
              f"| grip_r={traj16_dry[t,14]:.3f}, grip_l={traj16_dry[t,15]:.3f}")

---
## [선택] 실로봇 단일 Action Chunk 전송

> `Dry-run`으로 출력을 확인한 뒤, GR00T action chunk 1개를 실로봇으로 전송합니다.

- 기본값은 안전을 위해 `ENABLE_REAL_SINGLE_CHUNK = False` 입니다.
- 실행하려면 아래 코드 셀에서 `ENABLE_REAL_SINGLE_CHUNK = True`로 바꾼 뒤 실행하세요.
- `USE_REMOTE_GRIPPER=True`이고 `gripper_obj`가 존재하면 그리퍼 명령도 함께 전송합니다.

In [ ]:
# ---------- [선택] 실로봇 단일 action chunk 전송 ----------
if not globals().get("ENV_SETUP_DONE", False):
    raise RuntimeError("먼저 Step 3 셀을 실행하세요.")
if not globals().get("INITIAL_POSE_DONE", False):
    raise RuntimeError("먼저 Step 2 셀(초기 자세 이동)을 실행하세요.")

# 안전 게이트: 기본은 False (실행 시 True로 변경)
ENABLE_REAL_SINGLE_CHUNK = False

if not ENABLE_REAL_SINGLE_CHUNK:
    print("[real-chunk] ENABLE_REAL_SINGLE_CHUNK=False -> skip")
else:
    REAL_CHUNK_DT = 0.1
    REAL_CHUNK_HOLD_TIME = 0.2
    REAL_CHUNK_PRIORITY = ARM_COMMAND_PRIORITY

    # 안전 임계값 (필요 시 조정)
    SAFETY_MAX_TOTAL_NORM = 1.0   # chunk 전체 이동량 norm(rad)
    SAFETY_MAX_STEP_DELTA = 0.15  # step 간 최대 delta(rad)
    SAFETY_MAX_JOINT_DEG = 30.0   # 현재 위치 대비 최대 관절 변위(deg)
    SAFETY_OVERRIDE = True        # False면 위반 시 차단

    # 1) inference
    obs_real = env.get_observation()
    parsed_real = build_gr00t_input(obs_real, PROMPT, modality)
    action_chunk_real, _ = policy.get_action(parsed_real)
    traj16_real = flatten_action_chunk(action_chunk_real, action_keys)
    if traj16_real.ndim != 2 or traj16_real.shape[1] != 16:
        raise RuntimeError(f"Expected (T,16) action chunk, got {traj16_real.shape}")

    right_targets = traj16_real[:, 0:7]
    left_targets = traj16_real[:, 7:14]
    right_gripper = traj16_real[:, 14]
    left_gripper = traj16_real[:, 15]
    T = traj16_real.shape[0]
    print(f"[real-chunk] action chunk shape: {traj16_real.shape}")

    # 2) robot connect + current state
    robot_once = rby.create_robot(ROBOT_IP, "a") if hasattr(rby, "create_robot") else rby.create_robot_a(ROBOT_IP)
    robot_once.connect()
    assert robot_once.is_connected(), f"Failed to connect robot at {ROBOT_IP}"

    q0 = np.asarray(robot_once.get_state().position, dtype=np.float64)
    torso_hold = q0[2:8].copy()
    cur_right = q0[8:15].copy()
    cur_left = q0[15:22].copy()

    # 3) safety checks
    violations = []
    total_r = float(np.linalg.norm(right_targets[-1] - right_targets[0]))
    total_l = float(np.linalg.norm(left_targets[-1] - left_targets[0]))
    if total_r > SAFETY_MAX_TOTAL_NORM:
        violations.append(f"[A] right total norm {total_r:.4f} > {SAFETY_MAX_TOTAL_NORM:.2f}")
    if total_l > SAFETY_MAX_TOTAL_NORM:
        violations.append(f"[A] left total norm {total_l:.4f} > {SAFETY_MAX_TOTAL_NORM:.2f}")

    d_r = np.linalg.norm(np.diff(right_targets, axis=0), axis=1)
    d_l = np.linalg.norm(np.diff(left_targets, axis=0), axis=1)
    max_dr = float(d_r.max()) if d_r.size > 0 else 0.0
    max_dl = float(d_l.max()) if d_l.size > 0 else 0.0
    if max_dr > SAFETY_MAX_STEP_DELTA:
        violations.append(f"[B] right step delta {max_dr:.4f} > {SAFETY_MAX_STEP_DELTA:.2f}")
    if max_dl > SAFETY_MAX_STEP_DELTA:
        violations.append(f"[B] left step delta {max_dl:.4f} > {SAFETY_MAX_STEP_DELTA:.2f}")

    disp_r_deg = float(np.degrees(np.abs(right_targets - cur_right[None, :]).max()))
    disp_l_deg = float(np.degrees(np.abs(left_targets - cur_left[None, :]).max()))
    if disp_r_deg > SAFETY_MAX_JOINT_DEG:
        violations.append(f"[C] right joint disp {disp_r_deg:.2f}deg > {SAFETY_MAX_JOINT_DEG:.1f}deg")
    if disp_l_deg > SAFETY_MAX_JOINT_DEG:
        violations.append(f"[C] left joint disp {disp_l_deg:.2f}deg > {SAFETY_MAX_JOINT_DEG:.1f}deg")

    print("[real-chunk] safety summary")
    print(f"  total norm  : right={total_r:.4f}, left={total_l:.4f}")
    print(f"  max step d  : right={max_dr:.4f}, left={max_dl:.4f}")
    print(f"  max joint d : right={disp_r_deg:.2f}deg, left={disp_l_deg:.2f}deg")

    if violations and not SAFETY_OVERRIDE:
        if hasattr(robot_once, "disconnect"):
            robot_once.disconnect()
        raise RuntimeError("Safety check failed: " + " | ".join(violations))
    if violations and SAFETY_OVERRIDE:
        print("[real-chunk][WARN] safety violations (override on):")
        for v in violations:
            print("  -", v)

    # 4) send single chunk
    robot_once.power_on(".*")
    robot_once.servo_on(".*")
    robot_once.reset_fault_control_manager()
    if not robot_once.enable_control_manager():
        if hasattr(robot_once, "disconnect"):
            robot_once.disconnect()
        raise RuntimeError("[real-chunk] Failed to enable control manager")

    stream_once = robot_once.create_command_stream(priority=REAL_CHUNK_PRIORITY)
    send_errors = 0

    for i in range(T):
        cmd = rby.RobotCommandBuilder().set_command(
            rby.ComponentBasedCommandBuilder().set_body_command(
                rby.BodyComponentBasedCommandBuilder()
                .set_torso_command(
                    rby.JointPositionCommandBuilder()
                    .set_command_header(rby.CommandHeaderBuilder().set_control_hold_time(REAL_CHUNK_HOLD_TIME))
                    .set_position(torso_hold.astype(np.float64))
                    .set_minimum_time(REAL_CHUNK_DT)
                )
                .set_right_arm_command(
                    rby.JointPositionCommandBuilder()
                    .set_command_header(rby.CommandHeaderBuilder().set_control_hold_time(REAL_CHUNK_HOLD_TIME))
                    .set_position(right_targets[i].astype(np.float64))
                    .set_minimum_time(REAL_CHUNK_DT)
                )
                .set_left_arm_command(
                    rby.JointPositionCommandBuilder()
                    .set_command_header(rby.CommandHeaderBuilder().set_control_hold_time(REAL_CHUNK_HOLD_TIME))
                    .set_position(left_targets[i].astype(np.float64))
                    .set_minimum_time(REAL_CHUNK_DT)
                )
            )
        )

        try:
            stream_once.send_command(cmd)
        except RuntimeError as exc:
            send_errors += 1
            if "expired" in str(exc).lower():
                robot_once.wait_for_control_ready(1000)
                stream_once = robot_once.create_command_stream(priority=REAL_CHUNK_PRIORITY)
                stream_once.send_command(cmd)
            else:
                raise

        if USE_REMOTE_GRIPPER and globals().get("gripper_obj") is not None:
            try:
                gripper_obj.set_normalized_target(
                    np.array([float(right_gripper[i]), float(left_gripper[i])]),
                    wait_for_reply=False,
                )
            except Exception as ge:
                print(f"[real-chunk][WARN] gripper send failed: {ge}")

        if i == 0 or (i + 1) % 10 == 0 or (i + 1) == T:
            q_now = np.asarray(robot_once.get_state().position, dtype=np.float64)
            err_r = float(np.linalg.norm(right_targets[i] - q_now[8:15]))
            err_l = float(np.linalg.norm(left_targets[i] - q_now[15:22]))
            print(
                f"[real-chunk] step {i+1:3d}/{T} | "
                f"tracking_err(r={err_r:.4f}, l={err_l:.4f}) | "
                f"gripper(R={right_gripper[i]:+.3f}, L={left_gripper[i]:+.3f})"
            )

        time.sleep(REAL_CHUNK_DT)

    time.sleep(0.5)
    q_end = np.asarray(robot_once.get_state().position, dtype=np.float64)
    moved_r = float(np.linalg.norm(q_end[8:15] - cur_right))
    moved_l = float(np.linalg.norm(q_end[15:22] - cur_left))
    print(f"[real-chunk] done | moved norm right={moved_r:.4f}, left={moved_l:.4f}, send_errors={send_errors}")

    try:
        robot_once.cancel_control()
    except Exception:
        pass
    if hasattr(robot_once, "disconnect"):
        robot_once.disconnect()

---
## Step 5 — 실로봇 Inference Episode 루프 실행

GR00T inference → action chunk 실행을 반복하며 전체 에피소드를 수행합니다.

- **`EPISODE_LENGTH`**: 총 실행 스텝 수
- **`EXECUTE_CHUNK_SIZE`**: inference 1회당 실제 전송 스텝 수 (None = 전체 chunk)
- **`REPLAY_DT`**: 각 step 전송 간격(초)
- **`CONTROL_MODE`**: `"position"` 또는 `"impedance"`

### GR00T action 형식
`policy.get_action()` → `decode_action(use_relative_action=True)` → **absolute joint target (rad)**  
별도 변환 없이 `rby1_sdk` JointPositionCommandBuilder에 직접 전달합니다.

### 튜닝 가이드
- 움직임이 불안정하면 `EXECUTE_CHUNK_SIZE`를 줄이세요.
- 팔이 명령을 못 따라가면 `REPLAY_DT`를 키우세요.
- 처음에는 `EPISODE_LENGTH=50~100`으로 짧게 검증하세요.

In [ ]:
# ---------- Step 5: (실로봇) GR00T policy action replay — episode loop ----------
if not globals().get("ENV_SETUP_DONE", False):
    raise RuntimeError("먼저 Step 3 셀을 실행하세요.")
if not globals().get("INITIAL_POSE_DONE", False):
    raise RuntimeError("먼저 Step 2 셀(초기 자세 이동)을 실행하세요.")

# --------------------------------------------------
# 재생 파라미터
# --------------------------------------------------
REPLAY_DT          = 0.1
REPLAY_PRIORITY    = ARM_COMMAND_PRIORITY
REPLAY_HOLD_TIME   = 0.2

# --------------------------------------------------
# 제어 모드:  "position" | "impedance"
# --------------------------------------------------
CONTROL_MODE = "position"

ARM_STIFFNESS       = np.array([80.0, 80.0, 80.0, 80.0, 80.0, 80.0, 40.0])
ARM_DAMPING_RATIO   = 1.0
ARM_TORQUE_LIMIT    = np.array([30.0] * 7)
TORSO_STIFFNESS     = np.array([400.0] * 6)
TORSO_DAMPING_RATIO = 1.0
TORSO_TORQUE_LIMIT  = np.array([500.0] * 6)

assert CONTROL_MODE in ("position", "impedance"), f"Unknown CONTROL_MODE: {CONTROL_MODE}"

# --------------------------------------------------
# 로봇 연결
# --------------------------------------------------
robot = rby.create_robot(ROBOT_IP, "a") if hasattr(rby, "create_robot") else rby.create_robot_a(ROBOT_IP)
robot.connect()
assert robot.is_connected(), f"Failed to connect robot at {ROBOT_IP}"

robot.power_on(".*")
robot.servo_on(".*")
robot.reset_fault_control_manager()
if not robot.enable_control_manager():
    robot.disconnect()
    raise RuntimeError("[real-replay] Failed to enable control manager")

q_init     = np.asarray(robot.get_state().position, dtype=np.float64)
ep_start_r = q_init[8:15].copy()
ep_start_l = q_init[15:22].copy()
print(f"[real-replay] EPISODE_LENGTH={EPISODE_LENGTH}, CONTROL_MODE={CONTROL_MODE}")
print(f"[real-replay] EXECUTE_CHUNK_SIZE={EXECUTE_CHUNK_SIZE}")
print(f"[real-replay] episode start | right={np.round(ep_start_r, 4)}, left={np.round(ep_start_l, 4)}")

stream      = robot.create_command_stream(priority=REPLAY_PRIORITY)
total_steps = 0
chunk_idx   = 0
send_errors = 0

# --------------------------------------------------
# 로깅 버퍼
# --------------------------------------------------
_log_cmd_right    = []
_log_cmd_left     = []
_log_cmd_grip_r   = []
_log_cmd_grip_l   = []
_log_actual_right = []
_log_actual_left  = []
_log_actual_step  = []
_log_chunk_start  = []
_IMG_KEYS = ["observation/head_image", "observation/left_wrist_image", "observation/right_wrist_image"]
_log_obs_images = []

# --------------------------------------------------
# 커맨드 빌더 헬퍼
# --------------------------------------------------
def _build_arm_command(position, is_impedance):
    if is_impedance:
        return (
            rby.JointImpedanceControlCommandBuilder()
            .set_command_header(rby.CommandHeaderBuilder().set_control_hold_time(REPLAY_HOLD_TIME))
            .set_position(position.astype(np.float64))
            .set_minimum_time(REPLAY_DT)
            .set_stiffness(ARM_STIFFNESS)
            .set_damping_ratio(ARM_DAMPING_RATIO)
            .set_torque_limit(ARM_TORQUE_LIMIT)
        )
    else:
        return (
            rby.JointPositionCommandBuilder()
            .set_command_header(rby.CommandHeaderBuilder().set_control_hold_time(REPLAY_HOLD_TIME))
            .set_position(position.astype(np.float64))
            .set_minimum_time(REPLAY_DT)
        )

def _build_torso_command(position, is_impedance):
    if is_impedance:
        return (
            rby.JointImpedanceControlCommandBuilder()
            .set_command_header(rby.CommandHeaderBuilder().set_control_hold_time(REPLAY_HOLD_TIME))
            .set_position(position.astype(np.float64))
            .set_minimum_time(REPLAY_DT)
            .set_stiffness(TORSO_STIFFNESS)
            .set_damping_ratio(TORSO_DAMPING_RATIO)
            .set_torque_limit(TORSO_TORQUE_LIMIT)
        )
    else:
        return (
            rby.JointPositionCommandBuilder()
            .set_command_header(rby.CommandHeaderBuilder().set_control_hold_time(REPLAY_HOLD_TIME))
            .set_position(position.astype(np.float64))
            .set_minimum_time(REPLAY_DT)
        )

_use_impedance = (CONTROL_MODE == "impedance")

# --------------------------------------------------
# Episode loop: observe -> infer -> send (already absolute)
# --------------------------------------------------
while total_steps < EPISODE_LENGTH:
    remaining = EPISODE_LENGTH - total_steps

    # 1) observation + inference
    obs_ep = env.get_observation()

    _img_snap = {"step": total_steps}
    for _k in _IMG_KEYS:
        if _k in obs_ep:
            _img_snap[_k] = np.transpose(obs_ep[_k], (1, 2, 0))
    _log_obs_images.append(_img_snap)

    parsed_obs_ep = build_gr00t_input(obs_ep, PROMPT, modality)
    action_chunk_ep, _ = policy.get_action(parsed_obs_ep)

    # decode_action(use_relative_action=True) 내부에서 이미 absolute 변환 완료
    traj16 = flatten_action_chunk(action_chunk_ep, action_keys)

    right_targets     = traj16[:, 0:7]
    left_targets      = traj16[:, 7:14]
    right_gripper_out = traj16[:, 14]
    left_gripper_out  = traj16[:, 15]

    # 2) torso hold
    q_now      = np.asarray(robot.get_state().position, dtype=np.float64)
    torso_hold = q_now[2:8].copy()

    chunk_size    = traj16.shape[0]
    execute_size  = EXECUTE_CHUNK_SIZE if EXECUTE_CHUNK_SIZE is not None else chunk_size
    steps_to_send = min(execute_size, remaining)
    total_cmd_norm = float(np.linalg.norm(right_targets[steps_to_send-1] - right_targets[0]))

    _log_chunk_start.append(total_steps)
    print(
        f"[real-replay] chunk {chunk_idx+1} | "
        f"steps {total_steps+1}~{total_steps+steps_to_send}/{EPISODE_LENGTH} "
        f"(execute {steps_to_send}/{chunk_size}) | cmd_norm={total_cmd_norm:.4f}"
    )

    # 3) chunk 전송
    for i in range(steps_to_send):
        cmd = rby.RobotCommandBuilder().set_command(
            rby.ComponentBasedCommandBuilder().set_body_command(
                rby.BodyComponentBasedCommandBuilder()
                .set_torso_command(_build_torso_command(torso_hold, _use_impedance))
                .set_right_arm_command(_build_arm_command(right_targets[i], _use_impedance))
                .set_left_arm_command(_build_arm_command(left_targets[i], _use_impedance))
            )
        )

        try:
            stream.send_command(cmd)
        except RuntimeError as exc:
            send_errors += 1
            if "expired" in str(exc).lower():
                print(f"[real-replay][WARN] stream expired at step {total_steps}")
                robot.wait_for_control_ready(1000)
                stream = robot.create_command_stream(priority=REPLAY_PRIORITY)
                stream.send_command(cmd)
            else:
                raise

        if USE_REMOTE_GRIPPER and globals().get("gripper_obj") is not None:
            try:
                gripper_obj.set_normalized_target(
                    np.array([float(right_gripper_out[i]), float(left_gripper_out[i])]),
                    wait_for_reply=False,
                )
            except Exception as _ge:
                print(f"[real-replay][WARN] gripper send failed: {_ge}")

        _log_cmd_right.append(right_targets[i].copy())
        _log_cmd_left.append(left_targets[i].copy())
        _log_cmd_grip_r.append(float(right_gripper_out[i]))
        _log_cmd_grip_l.append(float(left_gripper_out[i]))

        if i == 0 or (i + 1) % 10 == 0 or (i + 1) == steps_to_send:
            q_diag  = np.asarray(robot.get_state().position, dtype=np.float64)
            err_r   = float(np.linalg.norm(right_targets[i] - q_diag[8:15]))
            err_l   = float(np.linalg.norm(left_targets[i]  - q_diag[15:22]))
            moved_r = float(np.linalg.norm(q_diag[8:15]  - ep_start_r))
            moved_l = float(np.linalg.norm(q_diag[15:22] - ep_start_l))
            _log_actual_right.append(q_diag[8:15].copy())
            _log_actual_left.append(q_diag[15:22].copy())
            _log_actual_step.append(total_steps + i)
            mark = " [IMP]" if _use_impedance else ""
            print(
                f"  step {total_steps+i+1:4d}/{EPISODE_LENGTH}{mark} "
                f"| tracking_err(r={err_r:.4f}, l={err_l:.4f}) "
                f"| moved(r={moved_r:.4f}, l={moved_l:.4f}) "
                f"| gripper(R={right_gripper_out[i]:+.3f}, L={left_gripper_out[i]:+.3f})"
            )

        time.sleep(REPLAY_DT)

    total_steps += steps_to_send
    chunk_idx   += 1

# --------------------------------------------------
# 최종 결과
# --------------------------------------------------
time.sleep(0.5)
q_after    = np.asarray(robot.get_state().position, dtype=np.float64)
obs_norm_r = float(np.linalg.norm(q_after[8:15]  - ep_start_r))
obs_norm_l = float(np.linalg.norm(q_after[15:22] - ep_start_l))

print(f"\n[real-replay] === episode done ({CONTROL_MODE} mode) ===")
print(f"[real-replay] total_steps={total_steps}, chunks={chunk_idx}, send_errors={send_errors}")
print(f"[real-replay] ep_start right : {np.round(ep_start_r, 4)}")
print(f"[real-replay] q_after  right : {np.round(q_after[8:15],  4)}")
print(f"[real-replay] ep_start left  : {np.round(ep_start_l, 4)}")
print(f"[real-replay] q_after  left  : {np.round(q_after[15:22], 4)}")
print(f"[real-replay] total movement norm | right={obs_norm_r:.4f}, left={obs_norm_l:.4f}")
print(f"[real-replay] log: {len(_log_cmd_right)} steps, {len(_log_chunk_start)} chunks")

try:
    robot.cancel_control()
except Exception:
    pass
if hasattr(robot, "disconnect"):
    robot.disconnect()

# [Initial Pose] (episode 후 복귀용)

In [ ]:
# ---------- Step 2: initial pose 이동 (safe path + head) ----------
if ENABLE_INITIAL_POSE:
    # data-collection 초기 자세
    TORSO_INIT_DEG = np.array([0.0, 20.0, -40.0, 35.0, 0.0, 0.0], dtype=np.float64)
    INIT_RIGHT_DEG = np.array([-24.0, -60.0, 10.0, -120.0, -60.0, 85.0, 0.0], dtype=np.float64)
    INIT_LEFT_DEG = np.array([-24.0, 60.0, -10.0, -120.0, 60.0, 85.0, 0.0], dtype=np.float64)
    TORSO_INIT = np.deg2rad(TORSO_INIT_DEG)
    INIT_RIGHT = np.deg2rad(INIT_RIGHT_DEG)
    INIT_LEFT = np.deg2rad(INIT_LEFT_DEG)

    MID1_RIGHT_DEG = np.array([70.0, -30.0, 0.0, -100.0, 0.0, -70.0, 0.0], dtype=np.float64)
    MID1_LEFT_DEG = np.array([70.0, 30.0, 0.0, -100.0, 0.0, -70.0, 0.0], dtype=np.float64)
    MID2_RIGHT_DEG = np.array([0.0, -15.0, 0.0, -100.0, 0.0, -70.0, 0.0], dtype=np.float64)
    MID2_LEFT_DEG = np.array([0.0, 15.0, 0.0, -100.0, 0.0, -70.0, 0.0], dtype=np.float64)
    MID1_RIGHT = np.deg2rad(MID1_RIGHT_DEG)
    MID1_LEFT = np.deg2rad(MID1_LEFT_DEG)
    MID2_RIGHT = np.deg2rad(MID2_RIGHT_DEG)
    MID2_LEFT = np.deg2rad(MID2_LEFT_DEG)

    TORSO_SLICE = slice(2, 8)
    RIGHT_SLICE = slice(8, 15)
    LEFT_SLICE = slice(15, 22)
    ELBOW_INIT_THRESHOLD_DEG = 90.0

    def _build_body_head_cmd(torso_q, right_q, left_q, head_q, dt):
        body_builder = rby.BodyComponentBasedCommandBuilder()
        if torso_q is not None:
            body_builder = body_builder.set_torso_command(
                rby.JointPositionCommandBuilder()
                .set_command_header(rby.CommandHeaderBuilder().set_control_hold_time(INIT_HOLD_TIME))
                .set_position(np.asarray(torso_q, dtype=np.float64))
                .set_minimum_time(float(dt))
            )
        body_builder = (
            body_builder
            .set_right_arm_command(
                rby.JointPositionCommandBuilder()
                .set_command_header(rby.CommandHeaderBuilder().set_control_hold_time(INIT_HOLD_TIME))
                .set_position(np.asarray(right_q, dtype=np.float64))
                .set_minimum_time(float(dt))
            )
            .set_left_arm_command(
                rby.JointPositionCommandBuilder()
                .set_command_header(rby.CommandHeaderBuilder().set_control_hold_time(INIT_HOLD_TIME))
                .set_position(np.asarray(left_q, dtype=np.float64))
                .set_minimum_time(float(dt))
            )
        )
        cbc = rby.ComponentBasedCommandBuilder().set_body_command(body_builder)
        if head_q is not None:
            cbc = cbc.set_head_command(
                rby.JointPositionCommandBuilder()
                .set_command_header(rby.CommandHeaderBuilder().set_control_hold_time(INIT_HOLD_TIME))
                .set_position(np.asarray(head_q, dtype=np.float64))
                .set_minimum_time(float(dt))
            )
        return rby.RobotCommandBuilder().set_command(cbc)

    def _move_waypoint_stream(robot, stream, name, torso_target, right_target, left_target, head_target, duration):
        q0 = np.asarray(robot.get_state().position, dtype=np.float64)
        start_torso = q0[TORSO_SLICE].copy()
        start_right = q0[RIGHT_SLICE].copy()
        start_left = q0[LEFT_SLICE].copy()
        goal_torso = torso_target if torso_target is not None else start_torso
        steps = max(2, int(np.ceil(duration / INIT_DT)))
        torso_path = np.linspace(start_torso, goal_torso, num=steps)
        right_path = np.linspace(start_right, right_target, num=steps)
        left_path = np.linspace(start_left, left_target, num=steps)
        for i in range(steps):
            cmd = _build_body_head_cmd(torso_path[i], right_path[i], left_path[i], head_target, dt=INIT_DT)
            try:
                stream.send_command(cmd)
            except RuntimeError as exc:
                if "expired" in str(exc).lower():
                    robot.wait_for_control_ready(1000)
                    stream = robot.create_command_stream(priority=INIT_COMMAND_PRIORITY)
                    stream.send_command(cmd)
                else:
                    raise
            time.sleep(INIT_DT)
        q_now = np.asarray(robot.get_state().position, dtype=np.float64)
        print(f"[init-pose] reached {name} | torso_now[1]={q_now[3]:+.4f}, right_now[0]={q_now[8]:+.4f}, left_now[0]={q_now[15]:+.4f}")
        return stream

    robot_init = rby.create_robot(ROBOT_IP, "a") if hasattr(rby, "create_robot") else rby.create_robot_a(ROBOT_IP)
    robot_init.connect()
    assert robot_init.is_connected(), f"Failed to connect robot at {ROBOT_IP}"

    robot_init.power_on(".*")
    robot_init.servo_on(".*")
    robot_init.enable_control_manager()

    try:
        robot_init.cancel_control()
    except Exception:
        pass
    robot_init.wait_for_control_ready(1000)

    q0 = np.asarray(robot_init.get_state().position, dtype=np.float64)
    print("[init-pose] current torso:", np.round(q0[TORSO_SLICE], 3))
    print("[init-pose] current right:", np.round(q0[RIGHT_SLICE], 3))
    print("[init-pose] current left :", np.round(q0[LEFT_SLICE], 3))
    if ENABLE_HEAD_INIT:
        print("[init-pose] target head(deg):", np.round(HEAD_INIT_DEG, 2))

    right_elbow_now = float(q0[RIGHT_SLICE][3])
    left_elbow_now  = float(q0[LEFT_SLICE][3])
    elbow_bent = (
        abs(right_elbow_now) > np.deg2rad(ELBOW_INIT_THRESHOLD_DEG)
        or abs(left_elbow_now) > np.deg2rad(ELBOW_INIT_THRESHOLD_DEG)
    )
    print(
        f"[init-pose] elbow check | right={np.degrees(right_elbow_now):+.1f}, "
        f"left={np.degrees(left_elbow_now):+.1f}  "
        f"-> {'near-INITIAL (bent)' if elbow_bent else 'near-ZERO (straight)'}"
    )

    if SAFE_INIT_PATH:
        if elbow_bent:
            waypoints = [
                ("midpoint2", None,       MID2_RIGHT, MID2_LEFT, 5.0),
                ("initial",   TORSO_INIT, INIT_RIGHT, INIT_LEFT, 3.0),
            ]
            print("[init-pose] near-initial detected: midpoint2 -> initial")
        else:
            waypoints = [
                ("midpoint1", None,       MID1_RIGHT, MID1_LEFT, 7.0),
                ("midpoint2", None,       MID2_RIGHT, MID2_LEFT, 7.0),
                ("initial",   TORSO_INIT, INIT_RIGHT, INIT_LEFT, 2.0),
            ]
            print("[init-pose] near-zero detected: midpoint1 -> midpoint2 -> initial")
    else:
        waypoints = [("initial", TORSO_INIT, INIT_RIGHT, INIT_LEFT, 4.0)]
        print("[init-pose] SAFE_INIT_PATH disabled: direct -> initial")

    head_target = HEAD_INIT if ENABLE_HEAD_INIT else None
    stream = robot_init.create_command_stream(priority=INIT_COMMAND_PRIORITY)
    for name, torso_target, right_target, left_target, duration in waypoints:
        print(f"[init-pose] move -> {name} (t={duration:.1f}s)")
        stream = _move_waypoint_stream(
            robot_init, stream, name=name,
            torso_target=torso_target, right_target=right_target,
            left_target=left_target, head_target=head_target, duration=duration,
        )

    time.sleep(0.2)
    q_end = np.asarray(robot_init.get_state().position, dtype=np.float64)
    err_t = float(np.linalg.norm(q_end[TORSO_SLICE] - TORSO_INIT))
    err_r = float(np.linalg.norm(q_end[RIGHT_SLICE] - INIT_RIGHT))
    err_l = float(np.linalg.norm(q_end[LEFT_SLICE] - INIT_LEFT))
    print(f"[init-pose] done | final error norm torso={err_t:.6f}, right={err_r:.6f}, left={err_l:.6f}")

    if USE_REMOTE_GRIPPER:
        for _arm in ("right", "left"):
            ok_v = robot_init.set_tool_flange_output_voltage(_arm, 12)
            print(f"[init-pose] tool flange 12V {_arm}: {'OK' if ok_v else 'FAILED'}")
        time.sleep(0.5)

    try:
        robot_init.cancel_control()
    except Exception:
        pass
    if hasattr(robot_init, "disconnect"):
        robot_init.disconnect()

    INITIAL_POSE_DONE = True
else:
    INITIAL_POSE_DONE = False
    print("[init-pose] skipped")

# [Gripper Initialization] (episode 후 재초기화용)

In [ ]:
# ---------- Step 2-5: Gripper 초기화 (homing + 열기) ----------
# 전제: Step 2 (initial pose) 실행 완료 -> tool flange 12V 공급 완료
if not globals().get("INITIAL_POSE_DONE", False):
    raise RuntimeError("먼저 Step 2 (initial pose) 셀을 실행하세요.")
if not globals().get("USE_REMOTE_GRIPPER", False):
    print("[gripper-init] USE_REMOTE_GRIPPER=False -> skip")
else:
    from rby1_remote_gripper import Gripper as RemoteGripper

    # Step 2에서 남아있는 stream 잔여 객체 제거
    # stream이 살아있으면 priority=10 control을 잡고 있어 이후 env 명령을 차단할 수 있음
    if "stream" in globals() and stream is not None:
        try:
            del stream
        except Exception:
            pass
        stream = None
        print("[gripper-init] 이전 stream 객체 정리 완료")

    # 이전 gripper 객체가 있으면 정리
    if "gripper_obj" in globals() and gripper_obj is not None:
        try:
            gripper_obj.stop()
            print("[gripper-init] 이전 gripper 객체 stop() 완료")
        except Exception as _e:
            print(f"[gripper-init] 이전 gripper stop 무시: {_e}")
        gripper_obj = None

    print("[gripper-init] RemoteGripper 연결 중...")
    gripper_obj = RemoteGripper()
    print(f"[gripper-init] host={gripper_obj.host}  port={gripper_obj.port}")

    if gripper_obj.host is None or gripper_obj.port is None:
        raise RuntimeError(
            "[gripper-init] gripper host/port가 설정되지 않았습니다.\n"
            "  scripts/deployment/config.yaml 또는 환경변수 확인"
        )

    # 1) ping / initialize
    print("[gripper-init] ping 확인 중...")
    ok_init = gripper_obj.initialize(verbose=True)
    print(f"[gripper-init] ping 결과: {'OK' if ok_init else 'FAILED'}")
    if not ok_init:
        raise RuntimeError("[gripper-init] 그리퍼 서버 응답 없음.")

    # 2) homing (범위 캘리브레이션)
    print("[gripper-init] homing 중... (30초 이내)")
    ok_homing = gripper_obj.homing()
    if not ok_homing:
        raise RuntimeError("[gripper-init] homing 실패")

    # homing 결과로 얻은 min_q / max_q 확인
    _min_q = getattr(gripper_obj, "min_q", None)
    _max_q = getattr(gripper_obj, "max_q", None)
    if _min_q is None or _max_q is None:
        raise RuntimeError(
            "[gripper-init] homing 성공했지만 min_q/max_q가 없음.\n"
            "  gripper_server.py 응답 형식에 min_q/max_q 필드가 있는지 확인하세요."
        )
    print(f"[gripper-init] homing 완료 | min_q={_min_q}  max_q={_max_q}")

    # 3) 제어 루프 시작
    print("[gripper-init] start() 호출 중...")
    gripper_obj.start()
    print("[gripper-init] start() 완료")

    # 4) 완전 열기/상태확인 구간 timeout 단축
    _old_timeout = getattr(gripper_obj, "timeout", None)
    _fast_timeout = 3.0
    try:
        _old_timeout_f = float(_old_timeout) if _old_timeout is not None else None
    except Exception:
        _old_timeout_f = None
    if _old_timeout_f is None or _old_timeout_f > _fast_timeout:
        gripper_obj.timeout = _fast_timeout
        try:
            gripper_obj._sock.settimeout(_fast_timeout)
        except Exception:
            pass
    print(f"[gripper-init] fast timeout 적용: {_old_timeout} -> {gripper_obj.timeout}")

    # 완전 열기 (normalized_q=1.0 -> OPEN) -- wait_for_reply=True로 확인
    print("[gripper-init] 완전 열기 명령 전송 (normalized=1.0)...")
    gripper_obj.set_normalized_target(np.array([1.0, 1.0]), wait_for_reply=True)
    time.sleep(0.5)

    # 현재 그리퍼 상태 확인
    try:
        _grip_state = gripper_obj.get_state()
        print(f"[gripper-init] 현재 그리퍼 상태: {_grip_state}  (열기 후 확인)")
    except Exception as _se:
        print(f"[gripper-init][WARN] 상태 조회 실패: {_se}")

    # 현재 normalized target 확인
    try:
        _grip_norm = gripper_obj.get_normalized_target()
        print(f"[gripper-init] 현재 normalized target: {_grip_norm}  (1.0=OPEN 이어야 함)")
    except Exception as _ne:
        print(f"[gripper-init][WARN] normalized target 조회 실패: {_ne}")

    GRIPPER_INIT_DONE = True
    print("[gripper-init] 완료")

---
## Action 로그 플롯 — 관절 궤적 및 Chunk 경계 분석

Episode loop 실행 후 `_log_*` 버퍼를 기반으로 3종류 그래프를 생성합니다.

| 그래프 | 내용 |
|--------|------|
| 전체 궤적 | 관절별 commanded vs actual (deg) |
| Step-to-step delta | chunk 경계 spike 확인 |
| Chunk 경계 zoom-in | 경계 전후 궤적 확대 |

In [ ]:
# ---------- Action chunk 로그 시각화 ----------
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

if "_log_cmd_right" not in globals() or len(_log_cmd_right) == 0:
    print("[plot] 로그 데이터 없음 — episode loop를 먼저 실행하세요.")
else:
    cmd_r    = np.array(_log_cmd_right)
    cmd_l    = np.array(_log_cmd_left)
    grip_r   = np.array(_log_cmd_grip_r)
    grip_l   = np.array(_log_cmd_grip_l)
    act_r    = np.array(_log_actual_right) if _log_actual_right else np.empty((0, 7))
    act_l    = np.array(_log_actual_left)  if _log_actual_left  else np.empty((0, 7))
    act_x    = np.array(_log_actual_step,  dtype=int)
    c_starts = np.array(_log_chunk_start,  dtype=int)

    N = cmd_r.shape[0]
    steps_x = np.arange(N)

    print(f"[plot] steps: {N}  |  chunks: {len(c_starts)}  |  actual samples: {len(act_x)}")

    def _plot_arm_full(title, cmd, act, act_steps, grip, tag):
        fig, axes = plt.subplots(2, 4, figsize=(20, 8), sharex=True)
        fig.suptitle(title, fontsize=13, fontweight="bold")
        axes_flat = axes.flatten()
        for j in range(7):
            ax = axes_flat[j]
            ax.plot(steps_x, np.degrees(cmd[:, j]), color="steelblue", lw=1.2, label="commanded")
            if len(act_steps) > 0:
                ax.scatter(act_steps, np.degrees(act[:, j]), color="tomato", s=18, zorder=5, label="actual")
            for cs in c_starts:
                ax.axvline(x=cs, color="dimgray", linestyle="--", lw=0.8, alpha=0.7)
            ax.set_title(f"{tag}_J{j}", fontsize=10)
            ax.set_ylabel("deg", fontsize=8)
            ax.tick_params(labelsize=7)
            ax.grid(True, alpha=0.3)
        ax_g = axes_flat[7]
        ax_g.plot(steps_x, grip, color="darkorange", lw=1.2)
        for cs in c_starts:
            ax_g.axvline(x=cs, color="dimgray", linestyle="--", lw=0.8, alpha=0.7)
        ax_g.set_title(f"{tag}_Gripper", fontsize=10)
        ax_g.set_ylabel("value", fontsize=8)
        ax_g.tick_params(labelsize=7)
        ax_g.grid(True, alpha=0.3)
        for ax in axes[1]:
            ax.set_xlabel("step", fontsize=8)
        handles = [
            mpatches.Patch(color="steelblue", label="commanded"),
            mpatches.Patch(color="tomato",    label="actual (sampled)"),
            plt.Line2D([0], [0], color="dimgray", linestyle="--", label="chunk start"),
        ]
        fig.legend(handles=handles, loc="lower center", ncol=3, fontsize=9, bbox_to_anchor=(0.5, -0.01))
        plt.tight_layout(rect=[0, 0.04, 1, 1])
        plt.show()

    _plot_arm_full("Right Arm (deg)", cmd_r, act_r, act_x, grip_r, "R")
    _plot_arm_full("Left Arm (deg)",  cmd_l, act_l, act_x, grip_l, "L")

    # Step-to-step delta norm
    delta_r = np.linalg.norm(np.diff(cmd_r, axis=0), axis=1)
    delta_l = np.linalg.norm(np.diff(cmd_l, axis=0), axis=1)

    fig2, (ax2r, ax2l) = plt.subplots(2, 1, figsize=(16, 6), sharex=True)
    fig2.suptitle("Step-to-step Command Delta (L2 norm, rad)", fontsize=11, fontweight="bold")
    for ax2, delta, color, arm_tag in [
        (ax2r, delta_r, "steelblue", "Right"),
        (ax2l, delta_l, "coral",     "Left"),
    ]:
        ax2.plot(np.arange(N - 1), delta, color=color, lw=0.9, label=f"{arm_tag} delta")
        for k, cs in enumerate(c_starts[1:]):
            ax2.axvline(x=cs - 1, color="red", linestyle="--", lw=1.5, alpha=0.85,
                        label="chunk boundary" if k == 0 else None)
        ax2.set_ylabel("delta norm (rad)", fontsize=9)
        ax2.set_title(f"{arm_tag} Arm", fontsize=10)
        ax2.grid(True, alpha=0.3)
        ax2.legend(fontsize=8)
    ax2l.set_xlabel("step", fontsize=9)
    plt.tight_layout()
    plt.show()

    print("\n[plot] === Step delta stats (chunk boundary vs internal) ===")
    for arm_tag, delta in [("Right", delta_r), ("Left", delta_l)]:
        in_mask = np.ones(len(delta), dtype=bool)
        for cs in c_starts[1:]:
            if 0 < cs - 1 < len(in_mask):
                in_mask[cs - 1] = False
        b_delta = delta[~in_mask]
        i_delta = delta[in_mask]
        print(
            f"  {arm_tag}: in-chunk avg={i_delta.mean():.4f} rad | "
            f"boundary avg={b_delta.mean() if len(b_delta) else 0:.4f} rad | "
            f"boundary max={b_delta.max()  if len(b_delta) else 0:.4f} rad"
        )

## Policy 입력 RGB 이미지 프레임 확인

Episode loop 실행 중 매 inference 시점에 캡처된 카메라 프레임을 시각화합니다.

- **행**: 카메라 종류 (Head / Left Wrist / Right Wrist)
- **열**: inference 시점 (chunk 인덱스)

In [ ]:
# ---------- Policy 입력 RGB 이미지 프레임 시각화 ----------
import matplotlib.pyplot as plt

if "_log_obs_images" not in globals() or len(_log_obs_images) == 0:
    print("[img-plot] 로그 데이터 없음 — episode loop를 먼저 실행하세요.")
else:
    cam_keys = [k for k in _IMG_KEYS if k in _log_obs_images[0]]
    n_snaps  = len(_log_obs_images)
    n_cams   = len(cam_keys)

    # 최대 12개 inference 시점만 표시
    MAX_COLS = 12
    step_indices = list(range(0, n_snaps, max(1, n_snaps // MAX_COLS)))[:MAX_COLS]

    fig, axes = plt.subplots(n_cams, len(step_indices),
                             figsize=(3 * len(step_indices), 3 * n_cams))
    if n_cams == 1:
        axes = axes[np.newaxis, :]
    if len(step_indices) == 1:
        axes = axes[:, np.newaxis]

    for col, si in enumerate(step_indices):
        snap = _log_obs_images[si]
        for row, ck in enumerate(cam_keys):
            ax = axes[row, col]
            if ck in snap:
                ax.imshow(snap[ck])
            ax.axis("off")
            if col == 0:
                ax.set_ylabel(ck.replace("observation/", ""), fontsize=9, rotation=0, labelpad=80, va="center")
            if row == 0:
                ax.set_title(f"chunk {si}\nstep={snap.get('step', '?')}", fontsize=8)

    fig.suptitle("Policy Input RGB Frames (per inference)", fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.show()
    print(f"[img-plot] {n_snaps} inference snapshots, showing {len(step_indices)}")

---
## Step 6 — 환경 및 정리

에피소드 종료 후 `RBY1Environment`를 안전하게 닫고 상태 플래그를 초기화합니다.

In [ ]:
# ---------- Step 6: 정리 ----------
if "env" not in globals():
    print("[stop] env 없음")
else:
    try:
        if env is not None:
            env.close()
            print("[cleanup] env 닫힘")
    except Exception as _e:
        print(f"[cleanup] env 닫기 실패: {_e}")
    env = None

if "gripper_obj" in globals() and gripper_obj is not None:
    try:
        gripper_obj.stop()
        print("[cleanup] gripper stopped")
    except Exception as _e:
        print(f"[cleanup] gripper stop 실패: {_e}")
    gripper_obj = None

ENV_SETUP_DONE = False
INITIAL_POSE_DONE = False
GRIPPER_INIT_DONE = False
print("[cleanup] done")

## Debug — 노이즈 원인 진단 (정규화/정책 분산/Chunk 경계)

아래 셀은 다음 3가지를 자동 점검합니다.

1. 서버가 실제로 로드한 체크포인트 경로와 processor/statistics 파일 일치 여부
2. 현재 실로봇 관측 state가 학습 통계(min/max) 범위를 벗어나는 비율
3. 동일 관측을 여러 번 넣었을 때 policy 출력 분산(샘플링 노이즈)

In [ ]:
# ---------- Debug: normalization/statistics + stochasticity diagnostics ----------
import hashlib
import json
from pathlib import Path

import numpy as np

if "policy" not in globals():
    raise RuntimeError("Step 1-2 셀을 먼저 실행하세요 (policy 필요).")
if "env" not in globals() or env is None:
    raise RuntimeError("Step 3 셀을 먼저 실행하세요 (env 필요).")

server_meta_dbg = policy.get_server_metadata()
ckpt_dir = Path(server_meta_dbg["checkpoint_dir"]).resolve()
print("=" * 80)
print("[debug] server checkpoint_dir:", ckpt_dir)
print("[debug] server embodiment_tag:", server_meta_dbg.get("embodiment_tag"))
print("[debug] server denoising steps:", server_meta_dbg.get("num_inference_timesteps"))

# 1) stats 파일 일치성 점검 (root vs processor/)
required_names = ["processor_config.json", "statistics.json", "embodiment_id.json"]
for name in required_names:
    p_root = ckpt_dir / name
    p_proc = ckpt_dir / "processor" / name
    if not p_root.exists():
        print(f"[debug][WARN] missing root file: {p_root}")
        continue
    h_root = hashlib.sha256(p_root.read_bytes()).hexdigest()
    if p_proc.exists():
        h_proc = hashlib.sha256(p_proc.read_bytes()).hexdigest()
        same = (h_root == h_proc)
        print(f"[debug] {name:<22} root==processor: {same}  sha256={h_root[:16]}...")
    else:
        print(f"[debug] {name:<22} processor copy not found (server may use root file directly)")

stats_path = ckpt_dir / "statistics.json"
if not stats_path.exists():
    raise RuntimeError(f"statistics.json not found under checkpoint_dir: {ckpt_dir}")

stats_all = json.loads(stats_path.read_text())
emb_key = server_meta_dbg.get("embodiment_tag", "new_embodiment")
if emb_key not in stats_all:
    raise RuntimeError(
        f"Embodiment '{emb_key}' not in statistics.json. Available={list(stats_all.keys())}"
    )
stats = stats_all[emb_key]
print(f"[debug] stats embodiment key: {emb_key}")

# 2) live state vs training min/max 범위 점검
print("\n" + "-" * 80)
print("[debug] live state range check against training min/max")

state_split = {
    "right_arm": slice(0, 7),
    "left_arm": slice(7, 14),
    "right_gripper": slice(14, 15),
    "left_gripper": slice(15, 16),
}

N_STATE_SAMPLES = 40
live_samples = {k: [] for k in state_split.keys()}
for _ in range(N_STATE_SAMPLES):
    obs_now = env.get_observation()
    s16 = np.asarray(obs_now["observation/state"], dtype=np.float32).reshape(-1)
    for k, sl in state_split.items():
        live_samples[k].append(s16[sl])

for k in state_split.keys():
    arr = np.stack(live_samples[k], axis=0)
    mn = np.asarray(stats["state"][k]["min"], dtype=np.float32)
    mx = np.asarray(stats["state"][k]["max"], dtype=np.float32)

    below = (arr < mn[None, :])
    above = (arr > mx[None, :])
    out = np.logical_or(below, above)
    out_ratio = float(out.mean())

    margin_lo = float(np.min(arr - mn[None, :]))
    margin_hi = float(np.min(mx[None, :] - arr))
    print(
        f"[state-range] {k:<14} out_ratio={out_ratio:7.4f}  "
        f"min_margin={margin_lo:+.5f}  max_margin={margin_hi:+.5f}"
    )

# 3) 동일 관측 반복 inference 시 샘플 분산(노이즈) 점검
print("\n" + "-" * 80)
print("[debug] repeated inference variance on identical observation")

obs_fixed = env.get_observation()
parsed_fixed = build_gr00t_input(obs_fixed, PROMPT, modality)

K_REPEATS = 8
traj_list = []
for _ in range(K_REPEATS):
    ac, _ = policy.get_action(parsed_fixed)
    traj_list.append(flatten_action_chunk(ac, action_keys))
traj_arr = np.stack(traj_list, axis=0)  # (K, T, 16)

std_all = np.std(traj_arr, axis=0)      # (T, 16)
mean_std_per_dim = np.mean(std_all, axis=0)

r_mean = float(np.mean(mean_std_per_dim[0:7]))
l_mean = float(np.mean(mean_std_per_dim[7:14]))
rg_mean = float(mean_std_per_dim[14])
lg_mean = float(mean_std_per_dim[15])
print(
    f"[repeat-var] mean std over horizon | right_arm={r_mean:.6f}, "
    f"left_arm={l_mean:.6f}, right_gripper={rg_mean:.6f}, left_gripper={lg_mean:.6f}"
)

# chunk boundary jump 진단 (episode 로그가 있을 때)
if "_log_cmd_right" in globals() and len(_log_cmd_right) > 2 and "_log_chunk_start" in globals():
    cmd_r = np.asarray(_log_cmd_right, dtype=np.float32)
    cmd_l = np.asarray(_log_cmd_left, dtype=np.float32)
    c_starts = np.asarray(_log_chunk_start, dtype=int)

    d_r = np.linalg.norm(np.diff(cmd_r, axis=0), axis=1)
    d_l = np.linalg.norm(np.diff(cmd_l, axis=0), axis=1)

    boundary_mask = np.zeros_like(d_r, dtype=bool)
    for cs in c_starts[1:]:
        if 0 < cs - 1 < len(boundary_mask):
            boundary_mask[cs - 1] = True

    for tag, delta in [("right", d_r), ("left", d_l)]:
        in_chunk = delta[~boundary_mask]
        at_boundary = delta[boundary_mask]
        b_mean = float(at_boundary.mean()) if at_boundary.size else 0.0
        i_mean = float(in_chunk.mean()) if in_chunk.size else 0.0
        ratio = (b_mean / i_mean) if i_mean > 1e-8 else np.inf
        print(
            f"[chunk-jump] {tag:<5} in_chunk_mean={i_mean:.6f}  "
            f"boundary_mean={b_mean:.6f}  ratio={ratio:.2f}"
        )
else:
    print("[chunk-jump] episode logs not available yet (run Step 5 first)")

print("=" * 80)
print("[debug] done")